# 11 — Avionics & Manufacturing

Complete avionics BOM, component placement validation, composite layup plan,
materials shopping list, and manufacturing procedure for the BWB UAV.

1. Avionics Bill of Materials
2. Component placement in fuselage (3D validation)
3. Electrical power budget
4. CG impact analysis (detailed BOM vs placeholder)
5. Composite layup plan
6. Structural mass from layup vs empirical
7. Materials shopping list
8. Manufacturing procedure

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyBboxPatch

from src.parameterization.design_variables import BWBParams, params_from_vector
from src.parameterization.bwb_aircraft import build_body_airfoil

# Load design
try:
    best_x = np.load('../output/best_x_v2.npy')
    params = params_from_vector(best_x)
    print('Loaded v2 optimized design')
except FileNotFoundError:
    params = BWBParams()
    print('Using default params')

## 1. Avionics Bill of Materials

In [ ]:
from src.systems.avionics import AvionicsSpec

avionics = AvionicsSpec.default_bwb()
avionics.print_bom()

## 2. Component Placement in Fuselage

In [ ]:
from src.systems.avionics import print_placement_summary, compute_component_positions_3d

print_placement_summary(params, avionics)

# XZ placement view
positions = compute_component_positions_3d(params, avionics)
bc = params.body_root_chord

af = build_body_airfoil(tc=params.body_tc_root, camber=params.body_camber,
                        reflex=params.body_reflex, le_droop=params.body_le_droop, n_pts=200)
af_x = af.coordinates[:, 0] * bc * 1000
af_z = af.coordinates[:, 1] * bc * 1000

fig, ax = plt.subplots(figsize=(16, 6))
ax.fill_between(af_x, af_z, alpha=0.1, color='#2c3e50')
ax.plot(af_x, af_z, 'k-', linewidth=2, label='Body envelope')

colors = {'FC': '#3498db', 'Receiver': '#3498db', 'GPS': '#27ae60', 'Pitot': '#e74c3c',
          'ESC': '#f39c12', 'BEC': '#f39c12', 'Camera FPV': '#9b59b6', 'VTX': '#9b59b6',
          'Servo': '#e67e22', 'LED': '#1abc9c', 'Current': '#f39c12'}

for p_info in positions:
    c = p_info
    if c['height_avail_mm'] < 1:
        continue
    x_mm, _, z_mm = c['position_mm']
    comp = [comp for comp in avionics.components if comp.name == c['name']][0]
    L, W, H = comp.dimensions_mm
    if H < 1:
        continue
    
    color = '#95a5a6'
    for key, col in colors.items():
        if key.lower() in c['name'].lower():
            color = col
            break
    
    rect = FancyBboxPatch((x_mm - L/2, z_mm - H/2), L, H,
                          boxstyle='round,pad=0.5', facecolor=color, alpha=0.6,
                          edgecolor='k', linewidth=0.8)
    ax.add_patch(rect)
    ax.text(x_mm, z_mm, c['name'][:8], ha='center', va='center', fontsize=6,
            fontweight='bold', color='white')

ax.set_xlabel('X [mm]')
ax.set_ylabel('Z [mm]')
ax.set_title('Component Placement — Body XZ Cross-Section', fontsize=14, fontweight='bold')
ax.set_aspect('equal')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('../output/avionics_placement_xz.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Electrical Power Budget

In [ ]:
budget = avionics.compute_power_budget()
print(f'Total avionics power: {budget["total_power_w"]:.1f} W')
print(f'BEC current draw:     {budget["bec_current_a"]:.2f} A / {budget["bec_capacity_a"]:.0f} A')
print(f'BEC margin:           {budget["bec_margin_pct"]:.0f}%')
if budget['bec_margin_pct'] < 20:
    print('WARNING: BEC margin is low, consider upgrading')
else:
    print('BEC margin OK')

## 4. CG Impact Analysis

Compare CG position with detailed avionics BOM vs the old 100g placeholder.

In [ ]:
from src.aero.evaluator import AeroEvaluator
from src.aero.mission import MissionCondition

mission = MissionCondition()
result = AeroEvaluator(mission, use_cg=True).evaluate(params)

print(f'Avionics mass (detailed BOM): {avionics.total_mass_g:.0f}g')
print(f'Avionics mass (old placeholder): 100g + 40g servos = 140g')
print(f'Delta: +{avionics.total_mass_g - 140:.0f}g')
print()
print(f'CG x/c:         {result.get("x_cg_frac", 0):.3f}')
print(f'Static margin:   {result["static_margin"]*100:.1f}% MAC')
print(f'L/D:             {result["L_over_D"]:.2f}')
print(f'T/D:             {result["T_over_D"]:.2f}')
print(f'Endurance:       {result["endurance_min"]:.1f} min')
print(f'Feasible:        {result["is_feasible"]}')
print()
if not result['is_feasible']:
    print('NOTE: Design may need CG adjustment (move battery forward)')
    print('Try reducing battery_xc from 0.22 to 0.15-0.18')

## 5. Composite Layup Plan

In [ ]:
from src.manufacturing.composite import ManufacturingPlan

plan = ManufacturingPlan.default_bwb(params)
plan.print_layup_summary()

## 6. Structural Mass: Layup vs Empirical

In [ ]:
plan.print_mass_breakdown()

## 7. Materials Shopping List

In [ ]:
plan.print_materials_bom()

print(f'Avionics cost:  {avionics.total_price_eur:.0f}\u20ac')
bom = plan.compute_materials_bom()
mat_cost = sum(item['price_eur'] for item in bom)
print(f'Materials cost:  {mat_cost:.0f}\u20ac')
print(f'TOTAL estimate: {avionics.total_price_eur + mat_cost:.0f}\u20ac (excl. tooling, consumables)')

## 8. Manufacturing Procedure

In [ ]:
from src.manufacturing.composite import print_manufacturing_steps
print_manufacturing_steps()